# Laboratorio Dirigido N.° 05
## Motor de decisión ante riesgo cambiario

**Curso:** Analítica Empresarial Integrada  
**Semana:** 5  
**Caso:** una empresa importadora peruana debe pagar **USD 100 000 dentro de 60 días**.  
**Fuente:** API pública de Series Estadísticas del Banco Central de Reserva del Perú.

### Pregunta orientadora
¿Cómo tomar una decisión defendible cuando no conocemos con certeza el tipo de cambio futuro?

> Este notebook se desarrolla íntegramente durante la clase con acompañamiento docente. No constituye una tarea, exposición ni entregable.

## Ruta de trabajo

1. Conexión y parámetros del caso.  
2. Descarga y trazabilidad de las series del BCRP.  
3. Limpieza y control de calidad.  
4. Selección explícita de la serie de venta.  
5. Retornos y volatilidad.  
6. Simulación Monte Carlo.  
7. Comparación entre comprar hoy y esperar.  
8. Motor transparente de decisión.  
9. Sensibilidad y reto en clase.

## BLOQUE 1 — Conexión, trazabilidad y calidad

### Celda 1 — Entorno de trabajo

In [1]:
# Librerías disponibles por defecto en Google Colab.
import sys
import platform
from datetime import date, timedelta

import numpy as np
import pandas as pd
import requests
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)

print("Python:", sys.version.split()[0])
print("Plataforma:", platform.platform())
print("Entorno listo.")

Python: 3.13.15
Plataforma: Linux-6.6.122+-x86_64-with-glibc2.39
Entorno listo.


### Celda 2 — Parámetros visibles y editables

In [2]:
# Parámetros empresariales
MONTO_USD = 100_000
HORIZONTE_CALENDARIO = 60
ESCENARIOS = 20_000
VENTANA_BASE = 252
SEMILLA = 2026

# Umbrales académicos del motor de decisión
TOLERANCIA_PROB_SOBRECOSTO = 0.40
TOLERANCIA_SOBRECOSTO_P95 = 0.02  # 2 % respecto de comprar hoy

# Series diarias del BCRP: compra y venta interbancaria (S/ por US$)
SERIES = ["PD04637PD", "PD04638PD"]
FECHA_FIN = date.today()
FECHA_INICIO = FECHA_FIN - timedelta(days=5 * 365)

print(f"Obligación: USD {MONTO_USD:,.0f}")
print(f"Horizonte: {HORIZONTE_CALENDARIO} días calendario")
print(f"Escenarios: {ESCENARIOS:,}")
print(f"Periodo solicitado: {FECHA_INICIO} a {FECHA_FIN}")

Obligación: USD 100,000
Horizonte: 60 días calendario
Escenarios: 20,000
Periodo solicitado: 2021-09-22 a 2026-09-21


### Celda 3 — Consulta a la API del BCRP

Se solicitan las dos series por su código oficial. La respuesta se valida antes de continuar.

In [3]:
BASE_API = "https://estadisticas.bcrp.gob.pe/estadisticas/series/api"
codigos = "-".join(SERIES)
URL = f"{BASE_API}/{codigos}/json/{FECHA_INICIO.isoformat()}/{FECHA_FIN.isoformat()}/esp"

respuesta = requests.get(URL, timeout=60)
print("Estado HTTP:", respuesta.status_code)
respuesta.raise_for_status()

datos_json = respuesta.json()
if "config" not in datos_json or "periods" not in datos_json:
    raise ValueError("La respuesta del BCRP no contiene 'config' y 'periods'.")

print("Series recibidas:", len(datos_json["config"]["series"]))
print("Periodos recibidos:", len(datos_json["periods"]))

Estado HTTP: 200
Series recibidas: 2
Periodos recibidos: 1304


### Celda 4 — Trazabilidad: nombres y códigos de las series

In [4]:
config_series = datos_json["config"]["series"]
trazabilidad = pd.DataFrame([
    {
        "codigo": s.get("code", SERIES[i] if i < len(SERIES) else ""),
        "nombre": s.get("name", ""),
        "unidad": s.get("unit", ""),
    }
    for i, s in enumerate(config_series)
])

display(trazabilidad)

if len(trazabilidad) != 2:
    raise ValueError("Se esperaban exactamente dos series: compra y venta.")

,codigo,nombre,unidad
0,PD04637PD,Tipo de cambio - TC Interbancario (S/ por US$)...,
1,PD04638PD,Tipo de cambio - TC Interbancario (S/ por US$)...,


### Celda 5 — Construcción del DataFrame y control de calidad

In [5]:
def convertir_valor(valor):
    if valor in (None, "", "n.d.", "n.d", "ND"):
        return np.nan
    return pd.to_numeric(str(valor).replace(",", ""), errors="coerce")

filas = []
for periodo in datos_json["periods"]:
    valores = periodo.get("values", [])
    if len(valores) != len(config_series):
        continue
    fila = {"periodo": periodo.get("name")}
    for i, serie in enumerate(config_series):
        nombre = serie.get("name", f"serie_{i+1}")
        fila[nombre] = convertir_valor(valores[i])
    filas.append(fila)

tc = pd.DataFrame(filas)
MESES_ES = {
    "Ene": "01", "Feb": "02", "Mar": "03", "Abr": "04",
    "May": "05", "Jun": "06", "Jul": "07", "Ago": "08",
    "Set": "09", "Sep": "09", "Oct": "10", "Nov": "11", "Dic": "12",
}

def fecha_bcrp(texto):
    partes = str(texto).strip().replace("-", ".").split(".")
    if len(partes) != 3:
        return pd.NaT
    dia, mes, anio = partes
    mes = MESES_ES.get(mes.title(), mes)
    anio = f"20{anio}" if len(anio) == 2 else anio
    return pd.to_datetime(f"{anio}-{mes}-{dia}", format="%Y-%m-%d", errors="coerce")

tc["fecha"] = tc["periodo"].map(fecha_bcrp)
tc = tc.drop(columns="periodo").sort_values("fecha").drop_duplicates("fecha")

columnas_tc = [c for c in tc.columns if c != "fecha"]
control = pd.DataFrame({
    "tipo": tc[columnas_tc].dtypes.astype(str),
    "valores_validos": tc[columnas_tc].notna().sum(),
    "valores_no_disponibles": tc[columnas_tc].isna().sum(),
    "minimo": tc[columnas_tc].min(),
    "maximo": tc[columnas_tc].max(),
})

print("Cobertura:", tc["fecha"].min().date(), "a", tc["fecha"].max().date())
display(control)
display(tc.tail())

Cobertura: 2021-09-22 a 2026-09-21


,tipo,valores_validos,valores_no_disponibles,minimo,maximo
Tipo de cambio - TC Interbancario (S/ por US$) - Compra,float64,1240,64,3.340857,4.134833
Tipo de cambio - TC Interbancario (S/ por US$) - Venta,float64,1240,64,3.342000,4.137500


,Tipo de cambio - TC Interbancario (S/ por US$) - Compra,Tipo de cambio - TC Interbancario (S/ por US$) - Venta,fecha
1299,3.374571,3.376000,2026-09-15
1300,3.364714,3.366714,2026-09-16
1301,3.362143,3.363714,2026-09-17
1302,NaN,NaN,2026-09-18
1303,NaN,NaN,2026-09-21


### Pausa guiada 1

**1. ¿Qué evidencia confirma que los datos proceden del BCRP?**
El estado HTTP de la respuesta es 200 (la celda 3 lo imprime y además hace `raise_for_status()`,
que detendría la ejecución si la API hubiera fallado). El JSON recibido trae las claves `config`
y `periods` con exactamente la forma que documenta la API pública del BCRP, y la tabla de
trazabilidad (Celda 4) muestra que los códigos devueltos coinciden con los códigos oficiales que
pedimos (`PD04637PD`, `PD04638PD`), con sus nombres y unidades declarados por la propia fuente. Es
decir, no solo "llegaron datos": llegaron los datos que pedimos, identificados por su código.

**2. ¿Por qué no debemos rellenar automáticamente los días con `n.d.`?**
Un `n.d.` significa que el BCRP no reportó valor ese día (feriado, no cotización, etc.). Rellenarlo
automáticamente (por ejemplo con el último valor conocido o con una interpolación) inventaría una
cifra que el Banco Central nunca publicó, y esa cifra inventada terminaría afectando el cálculo de
retornos y volatilidad como si fuera un dato real. Es preferible dejarlo como `NaN` y excluirlo
explícitamente (`dropna`), para que todo el análisis posterior se apoye únicamente en observaciones
que efectivamente ocurrieron.

**3. ¿Qué diferencia económica existe entre la serie compra y la serie venta?**
Son las dos puntas del *spread* cambiario del mercado interbancario: la serie **compra** es el
precio al que el banco compra dólares del público (le paga esa cantidad de soles por cada dólar),
y la serie **venta** es el precio al que el banco vende dólares (cobra esa cantidad de soles por
cada dólar). La empresa importadora del caso necesita *adquirir* dólares para pagar su obligación,
así que la serie relevante para ella es la de **venta** — que es, justamente, la que selecciona la
Celda 6 por nombre.

## BLOQUE 2 — Retornos y volatilidad

### Celda 6 — Selección de la serie de venta por su nombre

In [6]:
# Se selecciona por el nombre; nunca por la posición de la columna.
candidatas_venta = [c for c in columnas_tc if "venta" in c.lower()]
if len(candidatas_venta) != 1:
    raise ValueError(f"No se pudo identificar una única serie de venta: {candidatas_venta}")

col_venta = candidatas_venta[0]
candidatas_compra = [c for c in columnas_tc if "compra" in c.lower()]
col_compra = candidatas_compra[0] if len(candidatas_compra) == 1 else None

mercado = tc[["fecha", col_venta] + ([col_compra] if col_compra else [])].copy()
mercado = mercado.rename(columns={col_venta: "tc_venta", **({col_compra: "tc_compra"} if col_compra else {})})
mercado = mercado.dropna(subset=["fecha", "tc_venta"])

if "tc_compra" in mercado:
    inconsistencias = (mercado["tc_compra"] > mercado["tc_venta"]).sum()
    print("Filas con compra mayor que venta:", inconsistencias)

print("Serie principal:", col_venta)
display(mercado.tail())

Filas con compra mayor que venta: 1
Serie principal: Tipo de cambio - TC Interbancario (S/ por US$) - Venta


,fecha,tc_venta,tc_compra
1297,2026-09-11,3.367571,3.365571
1298,2026-09-14,3.380857,3.379000
1299,2026-09-15,3.376000,3.374571
1300,2026-09-16,3.366714,3.364714
1301,2026-09-17,3.363714,3.362143


### Celda 7 — Retornos logarítmicos y volatilidad

In [7]:
mercado["retorno_log"] = np.log(mercado["tc_venta"] / mercado["tc_venta"].shift(1))
retornos = mercado["retorno_log"].dropna()

vol_diaria = retornos.tail(VENTANA_BASE).std(ddof=1)
vol_anualizada = vol_diaria * np.sqrt(252)
tc_actual = mercado["tc_venta"].iloc[-1]

print(f"Tipo de cambio de venta actual: S/ {tc_actual:.4f} por US$")
print(f"Volatilidad diaria ({VENTANA_BASE} observaciones): {vol_diaria:.4%}")
print(f"Volatilidad anualizada referencial: {vol_anualizada:.2%}")

fig = px.line(mercado, x="fecha", y="tc_venta", title="Tipo de cambio interbancario venta")
fig.update_yaxes(title="S/ por US$")
fig.show()

Tipo de cambio de venta actual: S/ 3.3637 por US$
Volatilidad diaria (252 observaciones): 0.4204%
Volatilidad anualizada referencial: 6.67%


### Pausa guiada 2

Si la volatilidad sube pero el tipo de cambio *promedio* se mantiene parecido, el **valor esperado**
del costo casi no cambia — pero el **rango de resultados posibles** se ensancha en ambas direcciones:
aumenta tanto la probabilidad de un escenario muy favorable como la de uno muy adverso. Para una
empresa importadora esto importa porque su exposición no es simétrica en la práctica: un sobrecosto
grande puede comprometer el flujo de caja y el cumplimiento de la obligación de pago, mientras que un
ahorro igual de grande "solo" mejora el resultado, sin el mismo riesgo operativo. Por eso el motor de
decisión de este notebook no mira únicamente el costo esperado (promedio): mira también la
**probabilidad de sobrecosto** y el **P95**, que son los que realmente capturan el efecto de una
mayor volatilidad aunque el promedio no se mueva.

## BLOQUE 3 — Simulación Monte Carlo

### Celda 8 — Bootstrap histórico de retornos

In [8]:
def dias_habiles_aproximados(dias_calendario):
    return max(1, round(dias_calendario * 252 / 365))

def simular_tc_final(retornos_historicos, tc_inicial, horizonte_calendario,
                     escenarios=20_000, semilla=2026):
    horizonte_habil = dias_habiles_aproximados(horizonte_calendario)
    muestra = np.asarray(retornos_historicos.dropna(), dtype=float)
    if len(muestra) < 30:
        raise ValueError("No hay suficientes retornos históricos para simular.")
    rng = np.random.default_rng(semilla)
    caminos = rng.choice(muestra, size=(escenarios, horizonte_habil), replace=True)
    retornos_acumulados = caminos.sum(axis=1)
    tc_final = tc_inicial * np.exp(retornos_acumulados)
    return tc_final, horizonte_habil

retornos_base = retornos.tail(VENTANA_BASE)
tc_simulado, HORIZONTE_HABIL = simular_tc_final(
    retornos_base, tc_actual, HORIZONTE_CALENDARIO, ESCENARIOS, SEMILLA
)

resumen_tc = pd.Series(tc_simulado).describe(percentiles=[0.05, 0.50, 0.95])
print("Horizonte hábil aproximado:", HORIZONTE_HABIL)
display(resumen_tc.to_frame("tc_final_simulado"))

fig = px.histogram(
    x=tc_simulado, nbins=80,
    title="Distribución simulada del tipo de cambio al horizonte"
)
fig.add_vline(x=tc_actual, line_dash="dash", line_color="red", annotation_text="TC actual")
fig.update_xaxes(title="S/ por US$")
fig.update_yaxes(title="Frecuencia")
fig.show()

Horizonte hábil aproximado: 41


,tc_final_simulado
count,20000.000000
mean,3.341447
std,0.089860
min,2.867262
5%,3.189449
50%,3.343229
95%,3.485983
max,3.765520


### Pausa guiada 3

La distribución anterior no es un intervalo de confianza del tipo de cambio. Representa escenarios futuros simulados bajo el supuesto de que los retornos recientes son una referencia útil para el horizonte analizado.

## BLOQUE 4 — Comparación y motor de decisión

### Celda 9 — Comprar hoy frente a esperar

In [9]:
costo_comprar_hoy = MONTO_USD * tc_actual
costos_esperar = MONTO_USD * tc_simulado

costo_esperado = costos_esperar.mean()
prob_sobrecosto = np.mean(costos_esperar > costo_comprar_hoy)
p95_costo = np.percentile(costos_esperar, 95)
sobrecosto_esperado_pct = costo_esperado / costo_comprar_hoy - 1
sobrecosto_p95_pct = p95_costo / costo_comprar_hoy - 1

comparacion = pd.DataFrame({
    "indicador": [
        "Costo de comprar hoy", "Costo esperado si se espera",
        "Probabilidad de sobrecosto", "Percentil 95 del costo",
        "Sobrecosto esperado (%)", "Sobrecosto P95 (%)"
    ],
    "valor": [
        costo_comprar_hoy, costo_esperado, prob_sobrecosto,
        p95_costo, sobrecosto_esperado_pct, sobrecosto_p95_pct
    ]
})
display(comparacion)

,indicador,valor
0,Costo de comprar hoy,336371.428571
1,Costo esperado si se espera,334144.727842
2,Probabilidad de sobrecosto,0.406400
3,Percentil 95 del costo,348598.335749
4,Sobrecosto esperado (%),-0.006620
5,Sobrecosto P95 (%),0.036349


### Celda 10 — Tabla de riesgo e interpretación

In [10]:
tabla_riesgo = pd.DataFrame({
    "Métrica": ["TC actual", "TC esperado", "TC P95", "Costo hoy", "Costo esperado", "Costo P95", "Prob. sobrecosto"],
    "Resultado": [tc_actual, tc_simulado.mean(), np.percentile(tc_simulado, 95),
                  costo_comprar_hoy, costo_esperado, p95_costo, prob_sobrecosto]
})
display(tabla_riesgo)

print(
    f"Si la empresa espera, existe una probabilidad de {prob_sobrecosto:.1%} "
    f"de pagar más que si compra hoy. En el percentil 95, el costo alcanzaría "
    f"S/ {p95_costo:,.2f}."
)

,Métrica,Resultado
0,TC actual,3.363714
1,TC esperado,3.341447
2,TC P95,3.485983
3,Costo hoy,336371.428571
4,Costo esperado,334144.727842
5,Costo P95,348598.335749
6,Prob. sobrecosto,0.406400


Si la empresa espera, existe una probabilidad de 40.6% de pagar más que si compra hoy. En el percentil 95, el costo alcanzaría S/ 348,598.34.


### Celda 11 — Motor transparente de decisión

In [11]:
def motor_decision(prob_sobrecosto, sobrecosto_p95_pct,
                   tolerancia_prob=0.40, tolerancia_p95=0.02):
    razones = []
    if prob_sobrecosto > tolerancia_prob:
        razones.append(
            f"probabilidad de sobrecosto {prob_sobrecosto:.1%} > límite {tolerancia_prob:.1%}"
        )
    if sobrecosto_p95_pct > tolerancia_p95:
        razones.append(
            f"sobrecosto P95 {sobrecosto_p95_pct:.1%} > límite {tolerancia_p95:.1%}"
        )
    decision = "COMPRAR HOY" if razones else "ESPERAR"
    return decision, razones or ["los dos indicadores permanecen dentro de los umbrales definidos"]

decision_base, razones_base = motor_decision(
    prob_sobrecosto,
    sobrecosto_p95_pct,
    TOLERANCIA_PROB_SOBRECOSTO,
    TOLERANCIA_SOBRECOSTO_P95,
)

print("DECISIÓN DEL MOTOR:", decision_base)
print("Razones:")
for razon in razones_base:
    print("-", razon)

print("\nImportante: el motor aplica una política académica explícita; no reemplaza el juicio gerencial.")

DECISIÓN DEL MOTOR: COMPRAR HOY
Razones:
- probabilidad de sobrecosto 40.6% > límite 40.0%
- sobrecosto P95 3.6% > límite 2.0%

Importante: el motor aplica una política académica explícita; no reemplaza el juicio gerencial.


## BLOQUE 5 — Sensibilidad y robustez

### Celda 12 — Comparación de ventanas históricas

In [12]:
def evaluar_ventana(n):
    muestra = retornos.tail(n)
    simulados, _ = simular_tc_final(
        muestra, tc_actual, HORIZONTE_CALENDARIO, ESCENARIOS, SEMILLA
    )
    costos = MONTO_USD * simulados
    prob = np.mean(costos > costo_comprar_hoy)
    p95 = np.percentile(costos, 95)
    p95_pct = p95 / costo_comprar_hoy - 1
    decision, _ = motor_decision(
        prob, p95_pct,
        TOLERANCIA_PROB_SOBRECOSTO,
        TOLERANCIA_SOBRECOSTO_P95,
    )
    return {
        "ventana": n,
        "volatilidad_diaria": muestra.std(ddof=1),
        "costo_esperado": costos.mean(),
        "prob_sobrecosto": prob,
        "costo_p95": p95,
        "sobrecosto_p95_pct": p95_pct,
        "decision": decision,
    }

sensibilidad = pd.DataFrame([evaluar_ventana(252), evaluar_ventana(504)])
display(sensibilidad)

if sensibilidad["decision"].nunique() > 1:
    print("La decisión cambia: el resultado es sensible a la ventana histórica.")
else:
    print("La decisión no cambia entre ventanas, pero siguen existiendo supuestos no modelados.")

,ventana,volatilidad_diaria,costo_esperado,prob_sobrecosto,costo_p95,sobrecosto_p95_pct,decision
0,252,0.004204,334144.727842,0.40640,348598.335749,0.036349,COMPRAR HOY
1,504,0.003642,333383.516269,0.34585,345997.257447,0.028617,COMPRAR HOY


La decisión no cambia entre ventanas, pero siguen existiendo supuestos no modelados.


## Reto de aplicación en clase

La gerencia reduce su tolerancia máxima de probabilidad de sobrecosto a **25 %**. Modifique únicamente ese umbral, vuelva a ejecutar el motor y compare la decisión con la original.

In [13]:
TOLERANCIA_RETO = 0.25

decision_reto, razones_reto = motor_decision(
    prob_sobrecosto,
    sobrecosto_p95_pct,
    tolerancia_prob=TOLERANCIA_RETO,
    tolerancia_p95=TOLERANCIA_SOBRECOSTO_P95,
)

print("Decisión original:", decision_base)
print("Decisión con tolerancia de 25 %:", decision_reto)
print("Razones del nuevo resultado:")
for razon in razones_reto:
    print("-", razon)

Decisión original: COMPRAR HOY
Decisión con tolerancia de 25 %: COMPRAR HOY
Razones del nuevo resultado:
- probabilidad de sobrecosto 40.6% > límite 25.0%
- sobrecosto P95 3.6% > límite 2.0%


## Síntesis ejecutiva guiada

**Evidencia.** El tipo de cambio de venta actual es S/ 3.3637 por US$. Comprar hoy los
USD 100,000 cuesta S/ 336,371.43. Si la empresa espera 60 días, el costo esperado según la
simulación es S/ 334,144.73, con una probabilidad de 40.6% de terminar pagando más que si compra hoy,
y un costo en el percentil 95 de S/ 348,598.34.

**Inferencia.** La distribución simulada muestra que el resultado de esperar no es un único número,
sino un abanico de escenarios posibles construidos a partir de la volatilidad histórica reciente del
tipo de cambio (0.42% diaria sobre las últimas 252 observaciones, ~6.67% anualizada). El hecho de que
el costo esperado (promedio) sea *ligeramente menor* que comprar hoy (-0.66%) no implica que esperar
sea seguro: la cola derecha de la distribución (P95) muestra que, en un escenario adverso, el costo
podría llegar a ser 3.6% más caro que comprar hoy — ese es el riesgo real que un promedio favorable
esconde.

**Decisión.** El motor de decisión recomienda **COMPRAR HOY**, sustentado en al menos tres
indicadores: la probabilidad de sobrecosto (40.6%, que ya supera el umbral académico de 40%), el
sobrecosto en el percentil 95 (3.6%, muy por encima del umbral de 2%), y el hecho de que el ahorro
esperado (-0.66%) es marginal frente al riesgo de cola que muestran ambos indicadores.

**Condición de cambio.** La decisión cambiaría si la gerencia *aumentara* su tolerancia a la
probabilidad de sobrecosto por encima de 40.6% (en el "Reto de aplicación en clase" se probó lo
opuesto: bajarla a 25% solo refuerza la misma decisión), o si el umbral de sobrecosto P95 se
flexibilizara por encima de 3.6%. El análisis de sensibilidad (Celda 12) muestra que, incluso usando
una ventana histórica más larga (504 días, con menor volatilidad: 0.36% diaria), la decisión no
cambia: sigue siendo COMPRAR HOY, aunque con una probabilidad de sobrecosto algo menor (34.6%).

**Limitaciones.** Este análisis no incorpora, entre otros: (1) el *spread* cambiario efectivo que
cobraría el banco o casa de cambio al momento de la transacción real (se usó la serie de venta
interbancaria, no necesariamente el tipo de cambio al que la empresa transaría, y de hecho se detectó
1 fila donde compra > venta, una inconsistencia menor de la fuente que no se corrigió); (2) el costo
de financiamiento u oportunidad de mantener los soles invertidos durante los 60 días en lugar de
convertirlos hoy; y (3) instrumentos de cobertura cambiaria (forwards, opciones) que podrían fijar el
costo futuro sin asumir el riesgo completo de la simulación.

## Ticket de salida

En una frase, responda la pregunta orientadora e indique una evidencia concreta producida por el
notebook.

**Respuesta:**
Una decisión es defendible no porque adivine el tipo de cambio futuro, sino porque cuantifica el
riesgo con evidencia explícita: en este caso, una probabilidad de sobrecosto de 40.6% y un percentil
95 de S/ 348,598.34 frente al costo de comprar hoy (S/ 336,371.43), que es justamente la evidencia que
produce la celda del motor de decisión (Celda 11) de este notebook y que sustenta la recomendación de
comprar los dólares hoy.